In [3]:
import requests
import pandas as pd

# API Endpoint for fetching top 50 cryptocurrencies
API_URL = "https://api.coingecko.com/api/v3/coins/markets"
PARAMS = {
    "vs_currency": "usd",              # Get prices in USD
    "order": "market_cap_desc",         # Sort by market cap (highest first)
    "per_page": 50,                     # Fetch top 50 cryptocurrencies
    "page": 1,                           # Get first page results
    "sparkline": "false"                 # Disable sparkline data
}

# Function to fetch live cryptocurrency data
def fetch_crypto_data():
    response = requests.get(API_URL, params=PARAMS)
    if response.status_code == 200:
        return response.json()
    else:
        print("Error fetching data. Status code:", response.status_code)
        return None

# Fetch data
data = fetch_crypto_data()

if data:
    # Convert JSON response to Pandas DataFrame
    df = pd.DataFrame(data, columns=["name", "symbol", "current_price", "market_cap", "total_volume", "price_change_percentage_24h"])

    # Rename columns for better readability
    df.rename(columns={
        "name": "Cryptocurrency Name",
        "symbol": "Symbol",
        "current_price": "Current Price (USD)",
        "market_cap": "Market Capitalization",
        "total_volume": "24h Trading Volume",
        "price_change_percentage_24h": "24h Price Change (%)"
    }, inplace=True)

    # Save data to Excel
    df.to_excel("crypto_data.xlsx", index=False)
    print("Excel file 'crypto_data.xlsx' successfully created!")


Excel file 'crypto_data.xlsx' successfully created!


In [ ]:
import requests
import pandas as pd
import time
from openpyxl import Workbook
from fpdf import FPDF

def fetch_crypto_data():
    url = "https://api.coingecko.com/api/v3/coins/markets"
    params = {
        "vs_currency": "usd",
        "order": "market_cap_desc",
        "per_page": 50,
        "page": 1,
        "sparkline": "false"
    }
    response = requests.get(url, params=params)
    data = response.json()
    
    crypto_list = []
    for coin in data:
        crypto_list.append([
            coin['name'], 
            coin['symbol'].upper(),
            coin['current_price'],
            coin['market_cap'],
            coin['total_volume'],
            coin['price_change_percentage_24h']
        ])
    
    columns = ["Cryptocurrency Name", "Symbol", "Current Price (USD)", "Market Capitalization", "24h Trading Volume", "24h Price Change (%)"]
    df = pd.DataFrame(crypto_list, columns=columns)
    return df

def save_to_excel(df):
    filename = "crypto_data.xlsx"
    df.to_excel(filename, index=False)
    print(f"Data saved to {filename}")

def analyze_data(df):
    top_5 = df.nlargest(5, "Market Capitalization")
    avg_price = df["Current Price (USD)"].mean()
    highest_change = df.loc[df["24h Price Change (%)"].idxmax()]
    lowest_change = df.loc[df["24h Price Change (%)"].idxmin()]
    
    analysis_text = f"""
     **Top 5 Cryptocurrencies by Market Cap**
    {top_5[['Cryptocurrency Name', 'Market Capitalization']].to_string(index=False)}
    
     **Average Price of Top 50 Cryptos:** {avg_price:.2f} USD
    
    **Highest 24h Price Change:**
    {highest_change[['Cryptocurrency Name', '24h Price Change (%)']].to_string()}
    
     **Lowest 24h Price Change:**
    {lowest_change[['Cryptocurrency Name', '24h Price Change (%)']].to_string()}
    """
    
    with open("crypto_analysis_report.txt", "w") as file:
        file.write(analysis_text)
    print(" Analysis Report saved as 'crypto_analysis_report.txt'")

def generate_pdf_report():
    with open("crypto_analysis_report.txt", "r") as file:
        report_text = file.read()
    
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", size=12)
    pdf.multi_cell(0, 10, report_text)
    pdf.output("crypto_analysis_report.pdf")
    print(" PDF Report saved as 'crypto_analysis_report.pdf'")

if __name__ == "__main__":
    while True:
        print(" Fetching live crypto data...")
        df = fetch_crypto_data()
        save_to_excel(df)
        analyze_data(df)
        generate_pdf_report()
        print(" Waiting 5 minutes before next update...")
        time.sleep(300)  # 5 minutes


 Fetching live crypto data...
Data saved to crypto_data.xlsx
 Analysis Report saved as 'crypto_analysis_report.txt'
 PDF Report saved as 'crypto_analysis_report.pdf'
 Waiting 5 minutes before next update...
